[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/11_aprendizaje_contrastivo/11_proyecto_aprendizaje_contrastivo.ipynb)
# Proyecto 11 · Entrena el embedder de Políglota con negativos duros y mídelo contra baselines

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | ≈ 2.5 h |
| **Cómputo** | `FAST_DEV_RUN=True`: CPU/T4, ≈ 5 min. Modo completo (`FAST_DEV_RUN=False`, 10× pasos y corpus mayor): L4 recomendada; unidades ≈ horas reales × tarifa de tu sesión |
| **Requisitos** | Lección 11 |

## Contexto de negocio
Producto quiere que **Políglota Labs** deje de depender de embedders ajenos. Tu *tech lead* te pide un primer embedder propio ES/EN/código, entrenado desde cero con InfoNCE, **pares débiles** extraídos del corpus y las consultas etiquetadas de entrenamiento, y una ablación honesta: ¿ayudan los **negativos duros** minados con BM25 (con filtro de falsos negativos)? ¿Cómo queda frente a BM25 y frente a la torre sin entrenar?

**Conexión.** Usa el corpus/métricas del 00 y el pooling del 10; el embedder resultante se compara en el 12 con embedders modernos y se combina con BM25 en el 13.

## Dataset
Corpus Políglota (`carga_corpus`); consultas partidas ~50/50 en entrenamiento y test de forma determinista.

## Entregables y rúbrica
Umbrales calibrados con la solución de referencia (semilla 42, `FAST_DEV_RUN=True`, corpus sintético + stdlib: lo que pudo ejecutarse al construir el curso). Con datos reales, ejecuta la solución y usa sus números menos el margen.

| # | Entregable | Criterio |
|---|---|---|
| R1 | `info_nce(q, d, tau, d_extra, simetrica)` | coincide con la entropía cruzada calculada a mano (< 1e-5), con y sin negativos extra |
| R2 | `pares_entrenamiento(docs, q_train)` | ≥ 90 % de los documentos producen un par débil; incluye todos los pares etiquetados |
| R3 | `mina_duros(pares, perc=0.95)` | ningún negativo con score BM25 ≥ 0.95 × score del positivo; ≥ 50 % de los pares con ≥ 1 negativo |
| R4 | Embedder entrenado (solo negativos del lote) | nDCG@10 test ≥ torre sin entrenar + 0.10 |
| R5 | Variante con negativos duros | reportada; nDCG@10 test ≥ la de solo-lote − 0.15 (ver nota de calibración tras la solución) |
| R6 | Tabla final BM25 / sin entrenar / solo lote / + duros, y 4–5 frases de análisis | impresa |

In [ ]:
!pip install -q transformers sentence-transformers peft accelerate tokenizers datasets scipy scikit-learn matplotlib
import copy, json, math, os, random, re, time, warnings, zlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error(); transformers.utils.logging.disable_progress_bar()
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True   # True: CPU, modelos diminutos, ~10 min. False: más pasos/datos (GPU L4/A100 recomendada)
PAL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280", "morado": "#9467BD"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · torch {torch.__version__}")

### Utilidades (corpus, métricas, BM25 y bi-encoder de la lección). No hace falta modificarlas.

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
corpus = carga_corpus(FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids = [d["id"] for d in docs]
doc_txt = [d["text"] for d in docs]
txt_of = dict(zip(doc_ids, doc_txt))
# partición determinista de las consultas: ~50 % entrenamiento / ~50 % test
split_q = {q["qid"]: ("train" if zlib.crc32(q["qid"].encode()) % 2 == 0 else "test") for q in queries}
q_train = [q for q in queries if split_q[q["qid"]] == "train"]
q_test = [q for q in queries if split_q[q["qid"]] == "test"]
print("fuente:", corpus["source"], "| docs:", len(docs), "| queries train/test:", len(q_train), len(q_test))

In [ ]:
# ── Bi-encoder DESDE CERO: tokenizer BPE + Transformer encoder diminuto + mean pooling + L2 ──
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from transformers import PreTrainedTokenizerFast


def entrena_tokenizer(textos, vocab_size=2000):
    tk = Tokenizer(models.BPE(unk_token="[UNK]"))
    tk.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])
    tk.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Whitespace(), pre_tokenizers.Digits(individual_digits=True)])
    tk.train_from_iterator(textos, trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[PAD]", "[UNK]"]))
    return PreTrainedTokenizerFast(tokenizer_object=tk, pad_token="[PAD]", unk_token="[UNK]")


class BiEncoder(nn.Module):
    """Una sola torre (pesos compartidos para consulta y documento): emb -> Transformer -> media con máscara -> L2."""

    def __init__(self, V, d=128, capas=2, cabezas=4, max_len=64, dropout=0.1):
        super().__init__()
        self.emb, self.pos = nn.Embedding(V, d, padding_idx=0), nn.Embedding(max_len, d)
        capa = nn.TransformerEncoderLayer(d, cabezas, 2 * d, dropout=dropout, batch_first=True, norm_first=True)
        self.enc, self.ln, self.max_len = nn.TransformerEncoder(capa, capas, enable_nested_tensor=False), nn.LayerNorm(d), max_len

    def forward(self, ids, mask):
        x = self.emb(ids) + self.pos(torch.arange(ids.shape[1], device=ids.device))[None]
        h = self.ln(self.enc(x, src_key_padding_mask=~mask.bool()))
        m = mask.unsqueeze(-1).float()
        return F.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=-1)


def tokeniza(tok, textos, max_len=64):
    b = tok(textos, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return b["input_ids"].to(device), b["attention_mask"].to(device)


@torch.no_grad()
def codifica(model, tok, textos, bs=128):
    model.eval()
    return torch.cat([model(*tokeniza(tok, textos[i:i + bs])) for i in range(0, len(textos), bs)]).cpu()

In [ ]:
def alineacion(x, y, alpha=2):
    return (x - y).norm(dim=1).pow(alpha).mean().item()


def uniformidad(x, t=2):
    return torch.pdist(x, p=2).pow(2).mul(-t).exp().mean().log().item()

In [ ]:
def entrena_biencoder(pares, tok, pasos=300, B=32, tau=0.05, lr=2e-3, duros=None, sin_duplicados=True,
                      solo_positivos=False, simetrica=False, seed=SEED, grupo=None, log_cada=50, eval_fn=None, textos=None):
    """pares: lista de (texto_consulta, id_doc_positivo). duros: {índice_par: [ids de docs negativos]}.
    grupo: función par -> clave; sin_duplicados evita dos pares con la misma clave en un lote (falsos negativos).
    textos: dict id_doc -> texto (por defecto, el corpus Políglota)."""
    textos = textos or txt_of
    torch.manual_seed(seed); rng = random.Random(seed)
    model = BiEncoder(len(tok)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    grupo = grupo or (lambda p: p[1])
    hist = []
    for paso in range(pasos):
        orden = rng.sample(range(len(pares)), len(pares))
        lote, vistos = [], set()
        for i in orden:
            g = grupo(pares[i])
            if sin_duplicados and g in vistos:
                continue
            vistos.add(g); lote.append(i)
            if len(lote) == B:
                break
        q = model(*tokeniza(tok, [pares[i][0] for i in lote]))
        d = model(*tokeniza(tok, [textos[pares[i][1]] for i in lote]))
        extra = None
        if duros is not None:
            neg = [rng.choice(duros[i]) for i in lote if duros.get(i)]
            extra = model(*tokeniza(tok, [textos[n] for n in neg])) if neg else None
        if solo_positivos:
            loss = (q - d).pow(2).sum(1).mean()                 # sin negativos: ¡invita al colapso!
        else:
            loss = info_nce(q, d, tau, extra, simetrica)
        opt.zero_grad(); loss.backward(); opt.step()
        if paso % log_cada == 0 or paso == pasos - 1:
            reg = dict(paso=paso, loss=loss.item(), align=alineacion(q.detach(), d.detach()), unif=uniformidad(d.detach()))
            if eval_fn:
                reg["ndcg"] = eval_fn(model)
            hist.append(reg)
    return model.eval(), hist

In [ ]:
pol_tok = entrena_tokenizer(doc_txt + [q["text"] for q in queries], vocab_size=2000)
bm25_pol = BM25().fit([tokenize_simple(t) for t in doc_txt], doc_ids)
PASOS = 300 if FAST_DEV_RUN else 3000


def ndcg_test(model):
    D, Q = codifica(model, pol_tok, doc_txt), codifica(model, pol_tok, [q["text"] for q in q_test])
    run = {q["qid"]: rank_from_scores((Q @ D.T)[k].numpy(), doc_ids, 100) for k, q in enumerate(q_test)}
    return evaluate(run, {q["qid"]: qrels[q["qid"]] for q in q_test})["ndcg"]

## TODO 1 · `info_nce`
Positivo de `q[i]` = `d[i]`; los demás `d[j]` del lote son negativos; `d_extra` (M, dim) se añade como columnas; `simetrica=True` promedia con la dirección documento→consulta. `q` y `d` llegan ya normalizados.

<details><summary>🪜 Pista 1</summary>Es una entropía cruzada sobre la matriz de similitudes `q @ D.T / tau`, con etiquetas `0..B-1`.</details>
<details><summary>🪜 Pista 2</summary>`D = torch.cat([d, d_extra])`; `F.cross_entropy(q @ D.T / tau, torch.arange(B))`; para la simétrica, `F.cross_entropy(d @ q.T / tau, y)` (sin los extra).</details>

In [ ]:
def info_nce(q, d, tau=0.05, d_extra=None, simetrica=False):
    # TODO 1
    raise NotImplementedError


def test_info_nce():
    torch.manual_seed(0)
    q, d, h = (F.normalize(torch.randn(n, 8), dim=-1) for n in (4, 4, 3))
    S = q @ torch.cat([d, h]).T / 0.05
    manual = -(S[torch.arange(4), torch.arange(4)] - torch.logsumexp(S, 1)).mean()
    assert abs(info_nce(q, d, 0.05, h) - manual) < 1e-5
    S2 = q @ d.T / 0.05
    assert abs(info_nce(q, d, 0.05) + (torch.diag(S2) - torch.logsumexp(S2, 1)).mean()) < 1e-5
    return True


try:
    R1 = test_info_nce(); print("✅ R1")
except NotImplementedError:
    R1 = False; print("⏳ TODO 1 pendiente")

## TODO 2 · Pares de entrenamiento
- **Débiles**: para prosa, (primera frase, id del documento) si el documento tiene ≥ 3 frases; para código, (primera línea, id).
- **Etiquetados**: (texto de la consulta, id relevante) para cada consulta de `q_train`, **repetidos 5 veces** (son pocos y valiosos).

<details><summary>🪜 Pista 1</summary>`re.split(r"(?<=[.!?])\s+", texto)` parte en frases. Es la misma idea que CCPairs (título → cuerpo) a escala diminuta.</details>
<details><summary>🪜 Pista 2</summary>Recorre `docs`: si `d["lang"] == "code"`, el par es `(d["text"].split("\n")[0], d["id"])`; si no, parte en frases y añade `(frases[0], d["id"])` solo cuando hay ≥ 3. Los etiquetados: `[(q["text"], did) for q in q_train for did in qrels[q["qid"]]]`. Devuelve `(debiles + etiquetados * 5, len(debiles))`.</details>

In [ ]:
def pares_entrenamiento(docs, q_train):
    # TODO 2: devuelve (pares, n_debiles)
    raise NotImplementedError

## TODO 3 · Negativos duros con filtro TopK-PercPos
Para cada par, toma el top-30 de BM25 de su texto, quita el positivo, quita los candidatos con score ≥ `perc` × score del positivo (posibles falsos negativos) y quédate con hasta 10. Devuelve `{índice_del_par: [ids]}`.

<details><summary>🪜 Pista 1</summary>`sc = bm25_pol.scores(tokenize_simple(texto))`; el score del positivo es `sc[doc_ids.index(did)]` (precalcula un dict id → posición).</details>
<details><summary>🪜 Pista 2</summary>`umbral = perc * sc[pos_of[did]]`; recorre `np.argsort(-sc)[:k]` y conserva `doc_ids[j]` si `!= did` y `sc[j] < umbral`; corta a `[:n]`. El filtro es **relativo** al positivo (como en la sección 5 de la lección): un candidato casi tan bueno como el positivo es probablemente un falso negativo.</details>

In [ ]:
def mina_duros(pares, perc=0.95, k=30, n=10):
    # TODO 3
    raise NotImplementedError

## TODO 4 · Entrena, compara y comprueba la rúbrica
La celda está escrita: entrena la variante solo-lote y la variante con negativos duros (mismos pasos, $\tau=0.05$, $B=32$), mide nDCG@10 en test y evalúa la rúbrica. Después escribe tu análisis (4–5 frases): ¿por qué crees que los negativos duros ayudan o no en este corpus? Mira cuántos negativos por par sobreviven al filtro y qué tipo de documentos son.

In [ ]:
def experimento():
    pares, n_deb = pares_entrenamiento(docs, q_train)
    duros = mina_duros(pares)
    base = ndcg_test(BiEncoder(len(pol_tok)).to(device))
    bm = evaluate({q["qid"]: bm25_pol.search(tokenize_simple(q["text"]), 100) for q in q_test}, {q["qid"]: qrels[q["qid"]] for q in q_test})["ndcg"]
    m1, h1 = entrena_biencoder(pares, pol_tok, pasos=PASOS, B=32, tau=0.05)
    m2, h2 = entrena_biencoder(pares, pol_tok, pasos=PASOS, B=32, tau=0.05, duros=duros)
    r = dict(bm25=bm, sin_entrenar=base, solo_lote=ndcg_test(m1), con_duros=ndcg_test(m2),
             unif_lote=h1[-1]["unif"], unif_duros=h2[-1]["unif"])
    pos_of = {d: i for i, d in enumerate(doc_ids)}
    viol = 0
    for k_, cand in duros.items():
        sc = bm25_pol.scores(tokenize_simple(pares[k_][0]))
        viol += sum(sc[pos_of[c]] >= 0.95 * sc[pos_of[pares[k_][1]]] for c in cand)
    etiquetados = {(q["text"], did) for q in q_train for did in qrels[q["qid"]]}
    checks = {"R1 info_nce": R1,
              "R2 pares": n_deb >= 0.9 * len(docs) and etiquetados <= set(pares),
              "R3 duros filtrados": viol == 0 and np.mean([len(v) > 0 for v in duros.values()]) >= 0.5,
              "R4 entrenado ≥ sin entrenar + 0.10": r["solo_lote"] >= r["sin_entrenar"] + 0.10,
              "R5 con duros ≥ solo lote − 0.15": r["con_duros"] >= r["solo_lote"] - 0.15,
              "R6 tabla": True}
    print(f"pares={len(pares)} (débiles={n_deb}) | pares con ≥1 duro={np.mean([len(v) > 0 for v in duros.values()]):.0%}")
    for k_, v_ in r.items():
        print(f"   {k_:14s} {v_:.3f}")
    for k_, ok in checks.items():
        print(("✅ " if ok else "❌ ") + k_)
    return r


try:
    resultado = experimento()
except NotImplementedError:
    print("⏳ Completa los TODO 1–3.")

## Retos extra
1. **Curriculum**: entrena 150 pasos solo-lote y después 150 con negativos duros. ¿Cambia la conclusión?
2. **Negativos duros densos (ANCE)**: mina con tu propio modelo tras 150 pasos y re-entrena.
3. **Barrido de τ** ∈ {0.02, 0.05, 0.1, 0.2} y pérdida simétrica.
4. **sentence-transformers**: repite con `MultipleNegativesRankingLoss` sobre un modelo preentrenado (si tienes hub) usando un dataset `(anchor, positive, negative)` con tus negativos duros, y `CachedMultipleNegativesRankingLoss` con lote 256.
5. **Intervalos de confianza** (módulo 15): bootstrap por consulta de la diferencia solo-lote vs + duros.

## Reflexión
- ¿Qué parte de tu resultado se debe al tipo de benchmark (ítem conocido con fuerte solapamiento léxico)?
- ¿Cómo construirías consultas no léxicas para que el *golden set* premie lo que un embedder hace mejor que BM25?
- ¿Qué cambiarías para entrenar con millones de pares (muestreo, deduplicación, filtros, cómputo)?

---
## ⛔ SPOILER — intenta resolverlo primero
Solución de referencia completa.

In [ ]:
def info_nce(q, d, tau=0.05, d_extra=None, simetrica=False):
    D = d if d_extra is None else torch.cat([d, d_extra])
    y = torch.arange(q.shape[0], device=q.device)
    loss = F.cross_entropy(q @ D.T / tau, y)
    if simetrica:
        loss = (loss + F.cross_entropy(d @ q.T / tau, y)) / 2
    return loss


def pares_entrenamiento(docs, q_train):
    debiles = []
    for d in docs:
        if d["lang"] == "code":
            debiles.append((d["text"].split("\n")[0], d["id"]))
        else:
            fr = re.split(r"(?<=[.!?])\s+", d["text"].strip())
            if len(fr) >= 3:
                debiles.append((fr[0], d["id"]))
    etiquetados = [(q["text"], did) for q in q_train for did in qrels[q["qid"]]]
    return debiles + etiquetados * 5, len(debiles)


def mina_duros(pares, perc=0.95, k=30, n=10):
    pos_of = {d: i for i, d in enumerate(doc_ids)}
    duros = {}
    for i, (texto, did) in enumerate(pares):
        sc = bm25_pol.scores(tokenize_simple(texto))
        umbral = perc * sc[pos_of[did]]
        duros[i] = [doc_ids[j] for j in np.argsort(-sc)[:k] if doc_ids[j] != did and sc[j] < umbral][:n]
    return duros


R1 = test_info_nce()
resultado = experimento()

> **Nota de calibración.** Ejecución de referencia (fallback sintético + stdlib, 300 pasos en CPU): BM25 0.866 · sin entrenar 0.342 · solo lote 0.500 · con duros 0.533 (nDCG@10 test; revisión de ejecutabilidad oct-2026, idéntico en dos ejecuciones). En un prototipo con 400 pasos los duros quedaron, en cambio, 0.09 por **debajo**: el efecto es pequeño e inestable en este corpus, por eso R5 solo exige no perder más de 0.15. Hipótesis para tu análisis: los documentos sintéticos comparten plantillas y los «duros» que sobreviven al filtro son casi paráfrasis del positivo con otra herramienta o aspecto; con pocos pasos el modelo aún no domina los negativos fáciles; y la evaluación de ítem conocido premia el solapamiento léxico (BM25 sigue muy por encima). El umbral refleja lo medido, no lo deseado.